In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from fastmri.data.subsample import EquiSpacedMaskFunc
import glob
import os
from torchmetrics import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
import updated_dataloader

# ==========================================
# MODEL ARCHITECTURE (K-SPACE VERSION)
# ==========================================
class doubleConv2D(nn.Module):
    def __init__(self, input_ch, output_ch):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels=input_ch, out_channels=output_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=output_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=output_ch, out_channels=output_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=output_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)

class DownSample(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size=2, stride=2):
        super().__init__()
        self.downsample = nn.Sequential(
            nn.MaxPool2d(kernel_size=kernel_size, stride=stride),
            doubleConv2D(input_ch=input_channels, output_ch=output_channels)
        )
    
    def forward(self, x):
        return self.downsample(x)

class UNET_encoder(nn.Module):
    def __init__(self, **params):
        super().__init__()
        # Input is 30, but first layer projects to 48 (instead of 64)
        self.first_double_conv = doubleConv2D(30, 48) 
        self.downsample1 = DownSample(48, 96) 
        self.downsample2 = DownSample(96, 192)
        self.downsample3 = DownSample(192, 384)
        self.downsample4 = DownSample(384, 768)

    def forward(self, x):
        f1 = self.first_double_conv(x)
        f2 = self.downsample1(f1)
        f3 = self.downsample2(f2)
        f4 = self.downsample3(f3)
        bottleneck = self.downsample4(f4)
        return bottleneck, [f4, f3, f2, f1]

class UNET_decoder(nn.Module):
    def __init__(self, **params):
        super().__init__()
        self.tcv1 = nn.ConvTranspose2d(in_channels=768, out_channels=384, kernel_size=2, stride=2)
        self.double_conv1 = doubleConv2D(input_ch=768, output_ch=384) # 384 + 384 = 768 input
        
        self.tcv2 = nn.ConvTranspose2d(in_channels=384, out_channels=192, kernel_size=2, stride=2)
        self.double_conv2 = doubleConv2D(input_ch=384, output_ch=192) # 192 + 192 = 384 input
        
        self.tcv3 = nn.ConvTranspose2d(in_channels=192, out_channels=96, kernel_size=2, stride=2)
        self.double_conv3 = doubleConv2D(input_ch=192, output_ch=96)  # 96 + 96 = 192 input
        
        self.tcv4 = nn.ConvTranspose2d(in_channels=96, out_channels=48, kernel_size=2, stride=2)
        self.double_conv4 = doubleConv2D(input_ch=96, output_ch=48)   # 48 + 48 = 96 input

    def forward(self, bottleneck, skip_conns_list):
        upsample_1 = self.tcv1(bottleneck)
        concat_skip_1 = torch.concat([upsample_1, skip_conns_list[0]], dim=1)
        double_conv1_op = self.double_conv1(concat_skip_1)

        upsample_2 = self.tcv2(double_conv1_op)
        concat_skip_2 = torch.concat([upsample_2, skip_conns_list[1]], dim=1)
        double_conv2_op = self.double_conv2(concat_skip_2)

        upsample_3 = self.tcv3(double_conv2_op)
        concat_skip_3 = torch.concat([upsample_3, skip_conns_list[2]], dim=1)
        double_conv3_op = self.double_conv3(concat_skip_3)

        upsample_4 = self.tcv4(double_conv3_op)
        concat_skip_4 = torch.concat([upsample_4, skip_conns_list[3]], dim=1)
        double_conv4_op = self.double_conv4(concat_skip_4)

        return double_conv4_op

class UNET_final(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNET_encoder()
        self.decoder = UNET_decoder()
        # Final projection: 48 channels back down to 30
        self.one_Cross_one_conv = nn.Conv2d(in_channels=48, out_channels=30, kernel_size=1, stride=1)

    def forward(self, x):
        bottleneck, skip_conns_list = self.encoder(x)
        final_conv_op = self.decoder(bottleneck, skip_conns_list)
        final_kspace = self.one_Cross_one_conv(final_conv_op)
        return final_kspace

In [15]:
model = UNET_final()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("number of trainable params: ",trainable_params)

number of trainable params:  17478894


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from fastmri.data.subsample import EquiSpacedMaskFunc
import glob
import os
from torchmetrics import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
import updated_dataloader

# ==========================================
# MODEL ARCHITECTURE (K-SPACE VERSION)
# ==========================================
class doubleConv2D(nn.Module):
    def __init__(self, input_ch, output_ch):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels=input_ch, out_channels=output_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=output_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=output_ch, out_channels=output_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=output_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)

class DownSample(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size=2, stride=2):
        super().__init__()
        self.downsample = nn.Sequential(
            nn.MaxPool2d(kernel_size=kernel_size, stride=stride),
            doubleConv2D(input_ch=input_channels, output_ch=output_channels)
        )
    
    def forward(self, x):
        return self.downsample(x)
    
class UNET_encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.first_double_conv = doubleConv2D(30, 45)      # was 30→64
        self.downsample1 = DownSample(45, 90)               # was 64→128
        self.downsample2 = DownSample(90, 180)              # was 128→256
        self.downsample3 = DownSample(180, 360)             # was 256→512
        self.downsample4 = DownSample(360, 720)             # was 512→1024

    def forward(self, x):
        f1 = self.first_double_conv(x)
        f2 = self.downsample1(f1)
        f3 = self.downsample2(f2)
        f4 = self.downsample3(f3)
        bottleneck = self.downsample4(f4)
        return bottleneck, [f4, f3, f2, f1]


class UNET_decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tcv1 = nn.ConvTranspose2d(720, 360, kernel_size=2, stride=2)   # was 1024→512
        self.double_conv1 = doubleConv2D(720, 360)                           # was 1024→512
        self.tcv2 = nn.ConvTranspose2d(360, 180, kernel_size=2, stride=2)   # was 512→256
        self.double_conv2 = doubleConv2D(360, 180)                           # was 512→256
        self.tcv3 = nn.ConvTranspose2d(180, 90, kernel_size=2, stride=2)    # was 256→128
        self.double_conv3 = doubleConv2D(180, 90)                            # was 256→128
        self.tcv4 = nn.ConvTranspose2d(90, 45, kernel_size=2, stride=2)     # was 128→64
        self.double_conv4 = doubleConv2D(90, 45)                             # was 128→64

    def forward(self, bottleneck, skip_conns_list):
        upsample_1 = self.tcv1(bottleneck)
        concat_skip_1 = torch.concat([upsample_1, skip_conns_list[0]], dim=1)
        double_conv1_op = self.double_conv1(concat_skip_1)

        upsample_2 = self.tcv2(double_conv1_op)
        concat_skip_2 = torch.concat([upsample_2, skip_conns_list[1]], dim=1)
        double_conv2_op = self.double_conv2(concat_skip_2)

        upsample_3 = self.tcv3(double_conv2_op)
        concat_skip_3 = torch.concat([upsample_3, skip_conns_list[2]], dim=1)
        double_conv3_op = self.double_conv3(concat_skip_3)

        upsample_4 = self.tcv4(double_conv3_op)
        concat_skip_4 = torch.concat([upsample_4, skip_conns_list[3]], dim=1)
        double_conv4_op = self.double_conv4(concat_skip_4)

        return double_conv4_op


class UNET_final(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNET_encoder()
        self.decoder = UNET_decoder()
        self.one_Cross_one_conv = nn.Conv2d(45, 30, kernel_size=1, stride=1)  # was 64→30

    def forward(self, x):
        bottleneck, skip_conns_list = self.encoder(x)
        final_conv_op = self.decoder(bottleneck, skip_conns_list)
        final_kspace = self.one_Cross_one_conv(final_conv_op)
        return final_kspace

In [17]:
model = UNET_final()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("number of trainable params: ",trainable_params)

number of trainable params:  15363975


In [ ]:
class UNET_encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.first_double_conv = doubleConv2D(30, 48)       
        self.downsample1 = DownSample(48, 96)               
        self.downsample2 = DownSample(96, 184)               
        self.downsample3 = DownSample(184, 360)              
        self.downsample4 = DownSample(360, 720)              

    def forward(self, x):
        f1 = self.first_double_conv(x)    # 48 ch
        f2 = self.downsample1(f1)          # 96 ch
        f3 = self.downsample2(f2)          # 184 ch
        f4 = self.downsample3(f3)          # 360 ch
        bottleneck = self.downsample4(f4)  # 720 ch
        return bottleneck, [f4, f3, f2, f1]


class UNET_decoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 720 → concat with f4(360) → 720 → 360
        self.tcv1 = nn.ConvTranspose2d(720, 360, kernel_size=2, stride=2)
        self.double_conv1 = doubleConv2D(720, 360)

        # 360 → concat with f3(184) → 368 → 184
        self.tcv2 = nn.ConvTranspose2d(360, 184, kernel_size=2, stride=2)
        self.double_conv2 = doubleConv2D(368, 184)

        # 184 → concat with f2(96) → 192 → 96
        self.tcv3 = nn.ConvTranspose2d(184, 96, kernel_size=2, stride=2)
        self.double_conv3 = doubleConv2D(192, 96)

        # 96 → concat with f1(48) → 96 → 48
        self.tcv4 = nn.ConvTranspose2d(96, 48, kernel_size=2, stride=2)
        self.double_conv4 = doubleConv2D(96, 48)

    def forward(self, bottleneck, skip_conns_list):
        upsample_1 = self.tcv1(bottleneck)
        concat_skip_1 = torch.cat([upsample_1, skip_conns_list[0]], dim=1)  # 720
        double_conv1_op = self.double_conv1(concat_skip_1)

        upsample_2 = self.tcv2(double_conv1_op)
        concat_skip_2 = torch.cat([upsample_2, skip_conns_list[1]], dim=1)  # 368
        double_conv2_op = self.double_conv2(concat_skip_2)

        upsample_3 = self.tcv3(double_conv2_op)
        concat_skip_3 = torch.cat([upsample_3, skip_conns_list[2]], dim=1)  # 192
        double_conv3_op = self.double_conv3(concat_skip_3)

        upsample_4 = self.tcv4(double_conv3_op)
        concat_skip_4 = torch.cat([upsample_4, skip_conns_list[3]], dim=1)  # 96
        double_conv4_op = self.double_conv4(concat_skip_4)

        return double_conv4_op


class UNET_final(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNET_encoder()
        self.decoder = UNET_decoder()
        self.one_Cross_one_conv = nn.Conv2d(48, 30, kernel_size=1, stride=1)

    def forward(self, x):
        bottleneck, skip_conns_list = self.encoder(x)
        final_conv_op = self.decoder(bottleneck, skip_conns_list)
        final_kspace = self.one_Cross_one_conv(final_conv_op)
        return final_kspace

In [19]:
model = UNET_final()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("number of trainable params: ",trainable_params)

number of trainable params:  15512686
